In [1]:
import pandas as pd
import numpy as np

# CHARGEMENT
df = pd.read_csv("D:\\rouge_file\\Analyse_du_performance_de_la_bourse_casablanca\\csv_clean\\Physionomie_CLEAN.csv")
df = df[df['Type'] != 'Type'].drop_duplicates().copy()

print(f'Shape apres nettoyage basique : {df.shape}')
print(f'Nulls avant traitement        : {df.isnull().sum().sum()}')
print()

# COURS — 4218 nulls (1.02%)
# On laisse NaN intentionnellement
# Logique : les nulls de Cours appartiennent a deux categories :
#
# 1) Lignes agregats (MARCHE CENTRAL, ACTIONS, TOTAL GENERAL,
#    TRANSFERTS, DROITS, MARCHE DE BLOCS, OBLIGATIONS...) —
#    ces lignes representent des sommes de transactions a des
#    prix differents. Un agregat n a pas de cours unitaire.
#    Mettre un cours fictif n aurait aucun sens metier.
#
# 2) Titres individuels non echanges ce jour (TOTALENERGIES,
#    AKDITAL, TGCC...) — aucune transaction reelle ce jour.
#    Mettre le dernier cours connu fausserait les calculs
#    de volume et de rendement journalier.
#
# Ces nulls seront filtres lors des calculs de KPIs.
nb = df['Cours'].isnull().sum()
agregats = ['ACTIONS','MARCHE CENTRAL','TOTAL GENERAL','TRANSFERTS',
            'DROITS','MARCHE DE BLOCS','OBLIGATIONS','APPORTS DE TITRES',
            'AUGMENTATION DE CAPITAL EN NUMERAIRE']
nb_agregats = df[df['Instruments'].isin(agregats) & df['Cours'].isnull()].shape[0]
nb_titres   = nb - nb_agregats
print(f'[NaN CONSERVE] Cours : {nb} nulls conserves')
print(f'               dont {nb_agregats} lignes agregats (pas de cours unitaire)')
print(f'               dont {nb_titres} titres non echanges ce jour')
print()

# QUANTITE — 3 nulls (0.00%)
# Remplissage par 0
# Logique : 3 nulls sur 411 686 lignes — negligeable.
# Ces lignes correspondent a des agregats (APPORTS DE TITRES,
# ACTIONS, TOTAL GENERAL) avec Volume renseigne mais Quantite
# manquante. On remplace par 0 car impossible de reconstituer
# le nombre de titres depuis le volume seul sans le cours.
for col in ['Quantité', 'Quantite']:
    if col in df.columns:
        avant = df[col].isnull().sum()
        df[col] = df[col].fillna(0)
        apres  = df[col].isnull().sum()
        print(f'[ZERO] {col:15s} : {avant} nulls -> {apres} nulls restants')

print()

# RAPPORT FINAL
print('RAPPORT FINAL')
print(f'Shape finale         : {df.shape}')
print(f'Nulls restants total : {df.isnull().sum().sum()}')
nulls_restants = df.isnull().sum()
reste = nulls_restants[nulls_restants > 0]
if len(reste) > 0:
    print('\nDetail nulls restants (intentionnels) :')
    print(reste.to_string())
else:
    print('Aucun null restant — tableau 100% propre !')

df.to_csv('Physionomie_FINAL.csv', index=False, encoding='utf-8')
print(f'\nExporte : Physionomie_FINAL.csv — {len(df):,} lignes')

Shape apres nettoyage basique : (411686, 9)
Nulls avant traitement        : 4224

[NaN CONSERVE] Cours : 4218 nulls conserves
               dont 4073 lignes agregats (pas de cours unitaire)
               dont 145 titres non echanges ce jour

[ZERO] Quantité        : 3 nulls -> 0 nulls restants
[ZERO] Quantite        : 3 nulls -> 0 nulls restants

RAPPORT FINAL
Shape finale         : (411686, 9)
Nulls restants total : 4218

Detail nulls restants (intentionnels) :
Cours    4218

Exporte : Physionomie_FINAL.csv — 411,686 lignes
